# Volatility Smiles and Surface Reliability

Black-Scholes assumes one constant volatility, but real option markets usually imply different volatility levels across strikes and expiries. This notebook studies volatility smiles and checks how quote quality affects the reliability of the implied-volatility surface.

## Concepts

A volatility smile plots implied volatility against strike or moneyness for one expiry. Moneyness compares the underlying price to the strike, while log-moneyness uses \(\ln(K/S)\). Near-zero log-moneyness is near at-the-money. Negative values are strikes below spot, and positive values are strikes above spot.

A volatility surface extends the smile across multiple expiries. Raw surfaces can be noisy because stale quotes, zero bids, crossed markets, and wide bid-ask spreads can distort implied-volatility estimates.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

In [ ]:
from src.surface import (
    available_smile_slices,
    build_surface_grid,
    clean_vs_unclean_surface_comparison,
    prepare_surface_dataset,
    smile_residuals,
    surface_reliability_diagnostics,
)
from src.plotting import (
    plot_clean_vs_unclean_surface,
    plot_iv_bid_mid_ask_smile,
    plot_iv_mid_smiles,
    plot_surface_residuals,
    plot_volatility_surface_reliability,
)

## Load implied-volatility data

The input should include cleaned quote fields and bid, mid, and ask implied-volatility columns.

In [ ]:
input_path = project_root / "data" / "processed" / "iv_uncertainty_analysis.csv"

if not input_path.exists():
    input_path = project_root / "data" / "processed" / "iv_uncertainty_results.csv"

if not input_path.exists():
    raise FileNotFoundError("No IV uncertainty dataset found.")

iv_quotes = pd.read_csv(input_path)
surface_data = prepare_surface_dataset(iv_quotes)

surface_data.head(10)

## Available smile slices

In [ ]:
slice_summary = available_smile_slices(surface_data)
slice_summary

## Raw and cleaned surface grids

In [ ]:
raw_surface = build_surface_grid(surface_data, retained_only=False)
clean_surface = build_surface_grid(surface_data, retained_only=True)

raw_surface.head(10)

In [ ]:
clean_surface.head(10)

## Clean versus unclean surface comparison

In [ ]:
surface_comparison = clean_vs_unclean_surface_comparison(surface_data)
surface_comparison.head(15)

## Surface reliability diagnostics

The reliability score combines quote retention, IV completeness, median spread, and median IV uncertainty. It is not a pricing model. It is a data-quality score for deciding which parts of the surface are more trustworthy.

In [ ]:
diagnostics = surface_reliability_diagnostics(surface_data)
diagnostics

In [ ]:
diagnostics_path = project_root / "data" / "processed" / "surface_diagnostics.csv"
diagnostics.to_csv(diagnostics_path, index=False)

diagnostics_path

## Optional smile residuals

A simple fitted curve can help identify noisy points in one smile slice. Large residuals may point to local quote-quality issues or surface irregularities.

In [ ]:
residuals = smile_residuals(surface_data)
residuals[
    [
        "contractSymbol",
        "expiration",
        "option_type",
        "strike",
        "log_moneyness",
        "IV_mid",
        "fitted_IV_mid",
        "surface_residual",
    ]
].head(15)

## Save figures

In [ ]:
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

plot_iv_mid_smiles(
    surface_data,
    figures_dir / "iv_mid_smiles_by_expiry.png",
)

plot_iv_bid_mid_ask_smile(
    surface_data,
    figures_dir / "iv_bid_mid_ask_smile.png",
)

plot_clean_vs_unclean_surface(
    surface_data,
    figures_dir / "clean_vs_unclean_surface.png",
)

plot_volatility_surface_reliability(
    surface_data,
    figures_dir / "volatility_surface_reliability.png",
)

plot_surface_residuals(
    surface_data,
    figures_dir / "surface_residuals.png",
)

## Interpretation

The cleaned surface should usually be more stable than the raw surface because contracts with poor quote quality are removed from the retained set. The goal is not to force a perfect surface, but to show which parts of the implied-volatility surface are more reliable for downstream risk measures.